# Taxonomy Gap Analysis

How many tags in `test_annual.db` (facts table) do **not** appear anywhere in the 2025 GAAP taxonomy?

We check against all tags known to the taxonomy:
- `tag_info` from the Calculation sheet (all parent + child tags)
- `pres_parent` from the Presentation sheet (all tags in hierarchy)

A tag that appears in neither is a **custom** or **deprecated** XBRL tag that filers invented or carried over from older taxonomy versions.

In [1]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

# Point sec_core at the test mart DB
ROOT = Path("/home/chris/miniconda3/envs/tf/sec-data-core")
TEST_MART = ROOT / "test_db" / "test_annual.db"
assert TEST_MART.exists(), f"Test mart not found: {TEST_MART}"

os.environ["SEC_DATA_ROOT"] = str(ROOT)

In [2]:
# Load taxonomy — uses project-root ODS / parquet cache
from sec_core.taxonomy_loader import load_taxonomy, load_presentation_hierarchy

calc_map, calc_by_role, tag_info, parent_map, all_parents = load_taxonomy()
pres_descendants, pres_parent = load_presentation_hierarchy()

# All tags known to the taxonomy (Calculation + Presentation sheets combined)""
taxonomy_tags = set(tag_info.keys()) | set(pres_parent.keys())
print(f"Taxonomy tag universe: {len(taxonomy_tags):,} unique tags")
print(f"  From Calculation sheet (tag_info): {len(tag_info):,}")
print(f"  From Presentation sheet (pres_parent): {len(pres_parent):,}")

Taxonomy tag universe: 20,604 unique tags
  From Calculation sheet (tag_info): 5,747
  From Presentation sheet (pres_parent): 20,589


In [3]:
# Pull every distinct tag that appears in the mart facts table
conn = sqlite3.connect(f"file:{TEST_MART}?mode=ro", uri=True)

facts_tags_df = pd.read_sql_query(
    "SELECT tag, label, stmt, COUNT(*) AS n_rows, COUNT(DISTINCT cik) AS n_companies "
    "FROM facts GROUP BY tag, label, stmt ORDER BY tag",
    conn,
)
conn.close()

print(f"Distinct (tag, stmt) combos in mart: {len(facts_tags_df):,}")
print(f"Distinct tags in mart:               {facts_tags_df['tag'].nunique():,}")

Distinct (tag, stmt) combos in mart: 47,038
Distinct tags in mart:               41,777


In [4]:
# Classify each tag
facts_tags_df["in_taxonomy"] = facts_tags_df["tag"].isin(taxonomy_tags)
facts_tags_df["in_calc"]     = facts_tags_df["tag"].isin(tag_info.keys())
facts_tags_df["in_pres"]     = facts_tags_df["tag"].isin(pres_parent.keys())

summary = facts_tags_df.groupby("in_taxonomy")[["tag"]].nunique().rename(columns={"tag": "unique_tags"})
summary.index = summary.index.map({True: "In taxonomy", False: "NOT in taxonomy"})
summary["pct"] = (summary["unique_tags"] / summary["unique_tags"].sum() * 100).round(1)
print(summary.to_string())

                 unique_tags   pct
in_taxonomy                       
NOT in taxonomy        38110  91.2
In taxonomy             3667   8.8


## Tags missing from the taxonomy — detail

In [5]:
missing = (
    facts_tags_df[~facts_tags_df["in_taxonomy"]]
    .groupby("tag")
    .agg(
        label=("label", "first"),
        stmts=("stmt", lambda s: ", ".join(sorted(s.unique()))),
        total_rows=("n_rows", "sum"),
        n_companies=("n_companies", "max"),
    )
    .sort_values("n_companies", ascending=False)
    .reset_index()
)

print(f"{len(missing):,} tags in the mart have NO match in the 2025 GAAP taxonomy")
missing

38,110 tags in the mart have NO match in the 2025 GAAP taxonomy


,tag,label,stmts,total_rows,n_companies
0,IncreaseDecreaseInOperatingLeaseLiabilities,Current operating lease liabilities,CF,259,81
1,NoncashLeaseExpense,Non-cash Lease Expense,CF,139,58
2,NonCashLeaseExpense,Non Cash Lease Expense,CF,193,45
3,AccruedExpensesAndOtherCurrentLiabilities,Accrued Expenses And Other Current Liabilities,BS,120,37
4,NonCashInterestExpense,Non Cash Interest Expense,CF,65,25
...,...,...,...,...,...
38105,ImpairmentOfGoodwillAndOtherLongLivedAssets,Impairment of Goodwill and Other Long Lived As...,CF,2,1
38106,ImpairmentOfGoodwillOtherIntangiblesAndPropert...,"Impairment Of Goodwill, Other Intangibles And ...",CF,2,1
38107,ImpairmentOfInProcessResearchAndDevelopment,Impairment Of In Process Research And Development,"CF, IS",4,1
38108,ImpairmentOfIntangibleAndLongLivedAssets,Impairment Of Intangible And Long Lived Assets,CF,2,1


## Breakdown by statement

In [6]:
stmt_breakdown = (
    facts_tags_df.groupby(["stmt", "in_taxonomy"])["tag"]
    .nunique()
    .unstack(fill_value=0)
    .rename(columns={True: "in_taxonomy", False: "missing"})
)
stmt_breakdown["total"] = stmt_breakdown.sum(axis=1)
stmt_breakdown["pct_missing"] = (
    stmt_breakdown["missing"] / stmt_breakdown["total"] * 100
).round(1)
stmt_breakdown

in_taxonomy,missing,in_taxonomy,total,pct_missing
stmt,,,,
BS,6656,1446,8102,82.2
CF,25637,1843,27480,93.3
IS,7018,1144,8162,86.0


## Are the missing tags custom extensions?

Custom XBRL tags (invented by filers) are typically lowercase or contain a company-specific prefix. Standard `us-gaap` tags are PascalCase. A quick heuristic: does the tag start with a lowercase letter?

In [7]:
missing["likely_custom"] = missing["tag"].str[0].str.islower()

print("Missing tags — likely custom (starts lowercase):")
print(missing["likely_custom"].value_counts().to_string())
print()

# Show the non-custom ones (PascalCase, should be standard but missing from taxonomy)
non_custom = missing[~missing["likely_custom"]].copy()
print(f"\nPascalCase tags NOT in taxonomy ({len(non_custom)} tags — likely deprecated or removed):")
non_custom[["tag", "label", "stmts", "n_companies", "total_rows"]]

Missing tags — likely custom (starts lowercase):
likely_custom
False    38109
True         1


PascalCase tags NOT in taxonomy (38109 tags — likely deprecated or removed):


,tag,label,stmts,n_companies,total_rows
0,IncreaseDecreaseInOperatingLeaseLiabilities,Current operating lease liabilities,CF,81,259
1,NoncashLeaseExpense,Non-cash Lease Expense,CF,58,139
2,NonCashLeaseExpense,Non Cash Lease Expense,CF,45,193
3,AccruedExpensesAndOtherCurrentLiabilities,Accrued Expenses And Other Current Liabilities,BS,37,120
4,NonCashInterestExpense,Non Cash Interest Expense,CF,25,65
...,...,...,...,...,...
38105,ImpairmentOfGoodwillAndOtherLongLivedAssets,Impairment of Goodwill and Other Long Lived As...,CF,1,2
38106,ImpairmentOfGoodwillOtherIntangiblesAndPropert...,"Impairment Of Goodwill, Other Intangibles And ...",CF,1,2
38107,ImpairmentOfInProcessResearchAndDevelopment,Impairment Of In Process Research And Development,"CF, IS",1,4
38108,ImpairmentOfIntangibleAndLongLivedAssets,Impairment Of Intangible And Long Lived Assets,CF,1,2


## Coverage summary

In [8]:
total_facts = facts_tags_df["n_rows"].sum()
missing_facts = facts_tags_df.loc[~facts_tags_df["in_taxonomy"], "n_rows"].sum()

print("=" * 50)
print("COVERAGE SUMMARY")
print("=" * 50)
print(f"Total fact rows in mart:          {total_facts:>10,}")
print(f"Rows with non-taxonomy tags:      {missing_facts:>10,}  ({missing_facts/total_facts*100:.1f}%)")
print()
total_tags = facts_tags_df['tag'].nunique()
missing_tags = missing['tag'].nunique()
print(f"Total distinct tags in mart:      {total_tags:>10,}")
print(f"Tags not in taxonomy:             {missing_tags:>10,}  ({missing_tags/total_tags*100:.1f}%)")

COVERAGE SUMMARY
Total fact rows in mart:             855,654
Rows with non-taxonomy tags:          81,408  (9.5%)

Total distinct tags in mart:          41,777
Tags not in taxonomy:                 38,110  (91.2%)
